In [ ]:
# ============================================================================
# Experimental Suite: 20k Steps, TinyStories (BS=20, seq_len=128)
# Scheduler: Cosine with Hard Restarts Only
# ============================================================================
#
# Configuration Summary:
#   - Model: GPT-2-medium (345M)
#   - Dataset: TinyStories (roneneldan/TinyStories) ~20M tokens
#   - Steps: 20,000
#   - Batch Size: 20 (no gradient accumulation)
#   - Seq Len: 128
#   - Total Tokens: 20,000 × 20 × 128 ≈ 51.2M tokens ≈ 2.56 Epochs
#   - Scheduler: cosine_restarts (2 cycles)
#   - Learning Rate: 5e-4, Warmup: 100 steps
#
# Physics collected every 20 steps:
#   - v_norm, a_norm, mass, gamma, Re_old, Re_new (with EMA)
# ============================================================================

# 1. Install dependencies
!pip uninstall torchvision torchaudio -y
!pip install -U datasets huggingface_hub transformers accelerate -q

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    get_linear_schedule_with_warmup,
    get_constant_schedule,
    get_cosine_schedule_with_warmup,
    get_cosine_with_hard_restarts_schedule_with_warmup,
    get_polynomial_decay_schedule_with_warmup,
    default_data_collator
)
from datasets import load_dataset
from tqdm.notebook import tqdm
from collections import deque
import warnings
import gc
import math
from google.colab import files
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 2. Seed and device
# ------------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ------------------------------------------------------------
# 3. Custom inverse square root scheduler (not used here, but kept)
# ------------------------------------------------------------
from torch.optim.lr_scheduler import LambdaLR

def get_inverse_sqrt_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return math.sqrt(num_warmup_steps) / math.sqrt(max(current_step, num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

# ------------------------------------------------------------
# 4. Data preparation (TinyStories, seq_len=128)
# ------------------------------------------------------------
def prepare_data(model_name="gpt2"):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Load TinyStories (2023, ~20M tokens)
    dataset = load_dataset("roneneldan/TinyStories")
    # The dataset has only 'train' split, we split 10% for validation
    dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset_train = dataset["train"]
    dataset_valid = dataset["test"]

    # Use shorter sequence length (128) because stories are short
    max_seq_len = 128

    def tokenize(examples):
        return tokenizer(
            examples["text"], truncation=True, max_length=max_seq_len,
            padding="max_length", return_tensors="pt"
        )

    train_data = dataset_train.map(tokenize, batched=True, remove_columns=["text"])
    valid_data = dataset_valid.map(tokenize, batched=True, remove_columns=["text"])
    train_data.set_format("torch", columns=["input_ids", "attention_mask"])
    valid_data.set_format("torch", columns=["input_ids", "attention_mask"])

    return tokenizer, train_data, valid_data, max_seq_len

# ------------------------------------------------------------
# 5. Helper functions
# ------------------------------------------------------------
def get_flat_params(model):
    with torch.no_grad():
        return torch.cat([p.data.view(-1) for p in model.parameters()]).cpu()

def evaluate_loss(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            total_loss += outputs.loss.item()
            total_batches += 1
            if total_batches > 50: break
    model.train()
    return total_loss / total_batches

# ------------------------------------------------------------
# 6. Core training and data collection function
# ------------------------------------------------------------
def train_and_monitor_physics(learning_rate=5e-4, total_steps=20000,
                              record_interval=20, scheduler_type="cosine_restarts",
                              model_name="gpt2-medium"):
    """
    Train the model and record physical quantities.
    Batch size = 20, seq_len=128 => ~2.56 epochs over 20k steps on TinyStories.
    """
    # ---------- Model and data ----------
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    tokenizer, train_data, valid_data, max_seq_len = prepare_data(model_name)

    # --------------------------------------------
    # BATCH_SIZE = 20 (calculated for ~2.56 epochs in 20k steps on TinyStories)
    # --------------------------------------------
    BATCH_SIZE = 20
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=default_data_collator)
    valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=default_data_collator)

    # ---------- Optimizer and scheduler ----------
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.999))
    warmup_steps = 100
    beta1 = 0.9

    # We only use cosine_restarts here
    scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
        num_cycles=2)

    # ---------- Storage containers ----------
    param_queue = deque(maxlen=3)
    steps = []
    v_norm_list = []
    a_norm_list = []
    mass_history = []
    gamma_history = []
    train_losses = []
    val_losses = []
    val_step_markers = []

    # ---------- Training loop ----------
    step = 0
    pbar = tqdm(total=total_steps, desc=f"{model_name} | Adam | cosine_restarts | BS=20 (2.56 Epochs)",
                position=0, leave=True)

    while step < total_steps:
        for batch in train_loader:
            if step >= total_steps:
                break

            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

            # ---------- Physical quantities collection ----------
            if step % record_interval == 0:
                flat = get_flat_params(model)
                param_queue.append(flat)

                if len(param_queue) >= 3:
                    v_k = param_queue[1] - param_queue[0]
                    v_k1 = param_queue[2] - param_queue[1]
                    a_k = v_k1 - v_k

                    norm_v = torch.norm(v_k).item()
                    norm_a = torch.norm(a_k).item()

                    steps.append(step)
                    v_norm_list.append(norm_v)
                    a_norm_list.append(norm_a)

                    total_exp_avg_sq = 0.0
                    count = 0
                    for group in optimizer.param_groups:
                        for p in group['params']:
                            state = optimizer.state[p]
                            if 'exp_avg_sq' in state:
                                total_exp_avg_sq += torch.mean(state['exp_avg_sq']).item()
                                count += 1
                    if count > 0:
                        avg_sq = total_exp_avg_sq / count
                        eps = 1e-8
                        eta_eff = learning_rate / (math.sqrt(avg_sq) + eps)
                        mass_current = 1.0 / (eta_eff * (1 - beta1))
                        if len(mass_history) > 0:
                            gamma_current = mass_history[-1] - beta1 * mass_current
                        else:
                            gamma_current = 0.0
                    else:
                        mass_current = 1.0
                        gamma_current = 0.0
                    mass_history.append(mass_current)
                    gamma_history.append(gamma_current)

                    re_show = norm_a / (norm_v + 1e-12)
                    pbar.set_description(
                        f"{model_name} | Adam | cosine_restarts | BS=20 | Loss: {loss.item():.3f} | "
                        f"Re: {re_show:.3e}")

            # Validation every 500 steps
            if step % 500 == 0 and step > 0:
                val_loss = evaluate_loss(model, valid_loader)
                val_losses.append(val_loss)
                val_step_markers.append(step)

            step += 1
            pbar.update(1)

    pbar.close()

    # ---------- Offline computation ----------
    steps_arr = np.array(steps)
    v_norm_arr = np.array(v_norm_list, dtype=np.float32)
    a_norm_arr = np.array(a_norm_list, dtype=np.float32)
    mass_arr = np.array(mass_history, dtype=np.float32)
    gamma_arr = np.array(gamma_history, dtype=np.float32)

    with np.errstate(divide='ignore', invalid='ignore'):
        Re_old_raw = a_norm_arr / (v_norm_arr + 1e-12)
        Re_old_raw[np.isinf(Re_old_raw)] = 0.0
        Re_old_raw[np.isnan(Re_old_raw)] = 0.0

    with np.errstate(divide='ignore', invalid='ignore'):
        denom = gamma_arr * v_norm_arr + 1e-12
        Re_new_raw = mass_arr * a_norm_arr / denom
        Re_new_raw[np.isinf(Re_new_raw)] = 0.0
        Re_new_raw[np.isnan(Re_new_raw)] = 0.0

    alpha = 0.9
    ema_old = np.zeros_like(Re_old_raw)
    ema_new = np.zeros_like(Re_new_raw)
    if len(Re_old_raw) > 0:
        ema_old[0] = Re_old_raw[0]
        ema_new[0] = Re_new_raw[0]
        for i in range(1, len(Re_old_raw)):
            ema_old[i] = alpha * ema_old[i-1] + (1 - alpha) * Re_old_raw[i]
            ema_new[i] = alpha * ema_new[i-1] + (1 - alpha) * Re_new_raw[i]

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "steps": steps_arr,
        "v_norm": v_norm_arr,
        "a_norm": a_norm_arr,
        "Re_old_raw": Re_old_raw,
        "Re_old_ema": ema_old,
        "Re_new_raw": Re_new_raw,
        "Re_new_ema": ema_new,
        "mass_history": mass_arr,
        "gamma_history": gamma_arr,
        "train_loss": np.array(train_losses),
        "val_loss": np.array(val_losses),
        "val_steps": np.array(val_step_markers)
    }

# ============================================================================
# 7. Run single experiment: cosine_restarts on TinyStories
# ============================================================================
model_name = "gpt2-medium"
learning_rate = 5e-4
total_steps = 20000
record_interval = 20

exp_name = f"TinyStories_adam_lr5e-4_cosine_restarts_medium_20k_bs20"
print(f"\n{'='*60}")
print(f"Running: {exp_name} | Steps: {total_steps} | Batch: 20 (≈2.56 Epochs on TinyStories)")
print(f"{'='*60}")

exp_data = train_and_monitor_physics(
    learning_rate=learning_rate,
    total_steps=total_steps,
    record_interval=record_interval,
    scheduler_type="cosine_restarts",
    model_name=model_name
)

np.savez(f"re_data_{exp_name}.npz", **exp_data)
print(f"✅ Saved: re_data_{exp_name}.npz")

# ============================================================================
# 8. Package and download single file
# ============================================================================
print("\nPackaging npz file...")
!zip -r re_data_TinyStories_cosine_restarts_20k_bs20.zip *.npz

print("Initiating browser download...")
files.download("re_data_TinyStories_cosine_restarts_20k_bs20.zip")
print("All done! Cosine Restarts experiment on TinyStories completed.")

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.28 requires torchvision, which is not installed.
Device: cuda

Running: TinyStories_adam_lr5e-4_cosine_restarts_medium_20k_bs20 | Steps: 20000 | Batch: 20 (≈2.56 Epochs on TinyStories)


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.52GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


✅ Saved: re_data_TinyStories_adam_lr5e-4_cosine_restarts_medium_20k_bs20.npz

Packaging npz file...
  adding: re_data_TinyStories_adam_lr5e-4_cosine_restarts_medium_20k_bs20.npz (deflated 47%)
Initiating browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All done! Cosine Restarts experiment on TinyStories completed.


In [ ]:
# ============================================================================
# Experimental Suite: 20k Steps, TinyStories (BS=20, seq_len=128)
# Scheduler: Linear Decay with Warmup (polynomial decay, power=1)
# ============================================================================
#
# Configuration Summary:
#   - Model: GPT-2-medium (345M)
#   - Dataset: TinyStories (roneneldan/TinyStories) ~20M tokens
#   - Steps: 20,000
#   - Batch Size: 20 (no gradient accumulation)
#   - Seq Len: 128
#   - Total Tokens: 20,000 × 20 × 128 ≈ 51.2M tokens ≈ 2.56 Epochs
#   - Scheduler: linear decay (polynomial, power=1) with warmup
#   - Learning Rate: 5e-4, Warmup: 100 steps
#
# Physics collected every 20 steps:
#   - v_norm, a_norm, mass, gamma, Re_old, Re_new (with EMA)
# ============================================================================

# 1. Install dependencies (uncomment if running in Colab)
# !pip uninstall torchvision torchaudio -y
# !pip install -U datasets huggingface_hub transformers accelerate -q

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    get_linear_schedule_with_warmup,
    get_constant_schedule,
    get_cosine_schedule_with_warmup,
    get_cosine_with_hard_restarts_schedule_with_warmup,
    get_polynomial_decay_schedule_with_warmup,   # 用于线性衰减
    default_data_collator
)
from datasets import load_dataset
from tqdm.notebook import tqdm   # 如果不在 notebook 中运行，可改为 from tqdm import tqdm
from collections import deque
import warnings
import gc
import math
from google.colab import files   # 如果不在 Colab，请注释掉或替换为本地保存
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 2. Seed and device
# ------------------------------------------------------------
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ------------------------------------------------------------
# 3. Custom inverse square root scheduler (not used, but kept for reference)
# ------------------------------------------------------------
from torch.optim.lr_scheduler import LambdaLR

def get_inverse_sqrt_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return math.sqrt(num_warmup_steps) / math.sqrt(max(current_step, num_warmup_steps))
    return LambdaLR(optimizer, lr_lambda)

# ------------------------------------------------------------
# 4. Data preparation (TinyStories, seq_len=128)
# ------------------------------------------------------------
def prepare_data(model_name="gpt2"):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Load TinyStories (2023, ~20M tokens)
    dataset = load_dataset("roneneldan/TinyStories")
    # The dataset has only 'train' split, we split 10% for validation
    dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset_train = dataset["train"]
    dataset_valid = dataset["test"]

    # Use shorter sequence length (128) because stories are short
    max_seq_len = 128

    def tokenize(examples):
        return tokenizer(
            examples["text"], truncation=True, max_length=max_seq_len,
            padding="max_length", return_tensors="pt"
        )

    train_data = dataset_train.map(tokenize, batched=True, remove_columns=["text"])
    valid_data = dataset_valid.map(tokenize, batched=True, remove_columns=["text"])
    # 移除 set_format，避免触发 torchvision.io.VideoReader 导入错误（此处仅处理文本）
    # train_data.set_format("torch", columns=["input_ids", "attention_mask"])
    # valid_data.set_format("torch", columns=["input_ids", "attention_mask"])

    return tokenizer, train_data, valid_data, max_seq_len

# ------------------------------------------------------------
# 5. Helper functions
# ------------------------------------------------------------
def get_flat_params(model):
    with torch.no_grad():
        return torch.cat([p.data.view(-1) for p in model.parameters()]).cpu()

def evaluate_loss(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            total_loss += outputs.loss.item()
            total_batches += 1
            if total_batches > 50: break
    model.train()
    return total_loss / total_batches

# ------------------------------------------------------------
# 6. Core training and data collection function (modified for linear decay)
# ------------------------------------------------------------
def train_and_monitor_physics(learning_rate=5e-4, total_steps=20000,
                              record_interval=20, scheduler_type="linear_decay",
                              model_name="gpt2-medium"):
    """
    Train the model and record physical quantities.
    Batch size = 20, seq_len=128 => ~2.56 epochs over 20k steps on TinyStories.
    Scheduler: linear decay (polynomial with power=1) + warmup.
    """
    # ---------- Model and data ----------
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    tokenizer, train_data, valid_data, max_seq_len = prepare_data(model_name)

    # --------------------------------------------
    # BATCH_SIZE = 20 (calculated for ~2.56 epochs in 20k steps on TinyStories)
    # --------------------------------------------
    BATCH_SIZE = 20
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=default_data_collator)
    valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=default_data_collator)

    # ---------- Optimizer and scheduler ----------
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.999))
    warmup_steps = 100
    beta1 = 0.9

    # ---------- 根据 scheduler_type 选择调度器 ----------
    if scheduler_type == "cosine_restarts":
        scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
            num_cycles=2
        )
    elif scheduler_type == "linear_decay":
        scheduler = get_polynomial_decay_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
            lr_end=0.0,            # 最终学习率降为 0
            power=1.0              # 幂次为1 → 线性衰减
        )
    else:
        # 默认使用线性衰减
        scheduler = get_polynomial_decay_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
            lr_end=0.0,
            power=1.0
        )

    # ---------- Storage containers ----------
    param_queue = deque(maxlen=3)
    steps = []
    v_norm_list = []
    a_norm_list = []
    mass_history = []
    gamma_history = []
    train_losses = []
    val_losses = []
    val_step_markers = []

    # ---------- Training loop ----------
    step = 0
    pbar = tqdm(total=total_steps, desc=f"{model_name} | Adam | {scheduler_type} | BS=20 (2.56 Epochs)",
                position=0, leave=True)

    while step < total_steps:
        for batch in train_loader:
            if step >= total_steps:
                break

            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

            # ---------- Physical quantities collection ----------
            if step % record_interval == 0:
                flat = get_flat_params(model)
                param_queue.append(flat)

                if len(param_queue) >= 3:
                    v_k = param_queue[1] - param_queue[0]
                    v_k1 = param_queue[2] - param_queue[1]
                    a_k = v_k1 - v_k

                    norm_v = torch.norm(v_k).item()
                    norm_a = torch.norm(a_k).item()

                    steps.append(step)
                    v_norm_list.append(norm_v)
                    a_norm_list.append(norm_a)

                    total_exp_avg_sq = 0.0
                    count = 0
                    for group in optimizer.param_groups:
                        for p in group['params']:
                            state = optimizer.state[p]
                            if 'exp_avg_sq' in state:
                                total_exp_avg_sq += torch.mean(state['exp_avg_sq']).item()
                                count += 1
                    if count > 0:
                        avg_sq = total_exp_avg_sq / count
                        eps = 1e-8
                        eta_eff = learning_rate / (math.sqrt(avg_sq) + eps)
                        mass_current = 1.0 / (eta_eff * (1 - beta1))
                        if len(mass_history) > 0:
                            gamma_current = mass_history[-1] - beta1 * mass_current
                        else:
                            gamma_current = 0.0
                    else:
                        mass_current = 1.0
                        gamma_current = 0.0
                    mass_history.append(mass_current)
                    gamma_history.append(gamma_current)

                    re_show = norm_a / (norm_v + 1e-12)
                    pbar.set_description(
                        f"{model_name} | Adam | {scheduler_type} | BS=20 | Loss: {loss.item():.3f} | "
                        f"Re: {re_show:.3e}")

            # Validation every 500 steps
            if step % 500 == 0 and step > 0:
                val_loss = evaluate_loss(model, valid_loader)
                val_losses.append(val_loss)
                val_step_markers.append(step)

            step += 1
            pbar.update(1)

    pbar.close()

    # ---------- Offline computation ----------
    steps_arr = np.array(steps)
    v_norm_arr = np.array(v_norm_list, dtype=np.float32)
    a_norm_arr = np.array(a_norm_list, dtype=np.float32)
    mass_arr = np.array(mass_history, dtype=np.float32)
    gamma_arr = np.array(gamma_history, dtype=np.float32)

    with np.errstate(divide='ignore', invalid='ignore'):
        Re_old_raw = a_norm_arr / (v_norm_arr + 1e-12)
        Re_old_raw[np.isinf(Re_old_raw)] = 0.0
        Re_old_raw[np.isnan(Re_old_raw)] = 0.0

    with np.errstate(divide='ignore', invalid='ignore'):
        denom = gamma_arr * v_norm_arr + 1e-12
        Re_new_raw = mass_arr * a_norm_arr / denom
        Re_new_raw[np.isinf(Re_new_raw)] = 0.0
        Re_new_raw[np.isnan(Re_new_raw)] = 0.0

    alpha = 0.9
    ema_old = np.zeros_like(Re_old_raw)
    ema_new = np.zeros_like(Re_new_raw)
    if len(Re_old_raw) > 0:
        ema_old[0] = Re_old_raw[0]
        ema_new[0] = Re_new_raw[0]
        for i in range(1, len(Re_old_raw)):
            ema_old[i] = alpha * ema_old[i-1] + (1 - alpha) * Re_old_raw[i]
            ema_new[i] = alpha * ema_new[i-1] + (1 - alpha) * Re_new_raw[i]

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "steps": steps_arr,
        "v_norm": v_norm_arr,
        "a_norm": a_norm_arr,
        "Re_old_raw": Re_old_raw,
        "Re_old_ema": ema_old,
        "Re_new_raw": Re_new_raw,
        "Re_new_ema": ema_new,
        "mass_history": mass_arr,
        "gamma_history": gamma_arr,
        "train_loss": np.array(train_losses),
        "val_loss": np.array(val_losses),
        "val_steps": np.array(val_step_markers)
    }

# ============================================================================
# 7. Run single experiment: linear_decay on TinyStories
# ============================================================================
model_name = "gpt2-medium"
learning_rate = 5e-4
total_steps = 20000
record_interval = 20
scheduler_type = "linear_decay"   # 明确指定为线性衰减

exp_name = f"TinyStories_adam_lr5e-4_{scheduler_type}_medium_20k_bs20"
print(f"\n{'='*60}")
print(f"Running: {exp_name} | Steps: {total_steps} | Batch: 20 (≈2.56 Epochs on TinyStories)")
print(f"Scheduler: {scheduler_type} (polynomial decay with power=1)")
print(f"{'='*60}")

exp_data = train_and_monitor_physics(
    learning_rate=learning_rate,
    total_steps=total_steps,
    record_interval=record_interval,
    scheduler_type=scheduler_type,
    model_name=model_name
)

np.savez(f"re_data_{exp_name}.npz", **exp_data)
print(f"✅ Saved: re_data_{exp_name}.npz")

# ============================================================================
# 8. Package and download single file (only works in Colab)
# ============================================================================
print("\nPackaging npz file...")
!zip -r re_data_TinyStories_linear_decay_20k_bs20.zip *.npz   # 注意：这里打包所有npz，也可指定具体文件

print("Initiating browser download...")
files.download("re_data_TinyStories_linear_decay_20k_bs20.zip")
print("All done! Linear Decay experiment on TinyStories completed.")

Device: cuda

Running: TinyStories_adam_lr5e-4_linear_decay_medium_20k_bs20 | Steps: 20000 | Batch: 20 (≈2.56 Epochs on TinyStories)
Scheduler: linear_decay (polynomial decay with power=1)


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Map:   0%|          | 0/211972 [00:00<?, ? examples/s]

gpt2-medium | Adam | linear_decay | BS=20 (2.56 Epochs):   0%|          | 0/20000 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


✅ Saved: re_data_TinyStories_adam_lr5e-4_linear_decay_medium_20k_bs20.npz

Packaging npz file...
  adding: re_data_TinyStories_adam_lr5e-4_linear_decay_medium_20k_bs20.npz (deflated 48%)
Initiating browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All done! Linear Decay experiment on TinyStories completed.
